# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [ ]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


In [1]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [2]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
# jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [3]:
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
# jugar_frozen_lake(env)

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [4]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 2


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [5]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 1.0%
Recompensa Promedio: 0.010
Desviación Estándar: 0.099
Total de Victorias: 1


La siguiente celda define una función para entrenar un agente.

In [6]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=100, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
            
            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")


✓ Funciones de entrenamiento definidas


## 1. **Agente Q-Learning**
La siguiente celda define el agente de Q-Learning a implementar.

In [43]:
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self, espacio_observacion, espacio_acciones, alpha_offset=25, alpha_min=0.03,
                 gamma=0.99, epsilon=1, epsilon_decay=0.000015, epsilon_min=0.01):
        """
        Inicializa el agente Q-Learning.

        Args:
            espacio_observacion (int): Número de estados posibles.
            espacio_acciones (int): Número de acciones posibles.
            alpha_offset (float): Offset para el cálculo de alpha dinámico.
            alpha_min (float): Valor mínimo de alpha.
            gamma (float): Factor de descuento.
            epsilon (float): Probabilidad inicial de exploración (epsilon-greedy).
            epsilon_decay (float): Cantidad fija a disminuir epsilon por episodio.
            epsilon_min (float): Valor mínimo que puede tomar epsilon en el entrenamiento.
        """
        self.n = int(np.sqrt(espacio_observacion))
        self.qtable = np.zeros((espacio_observacion, espacio_acciones)) 
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        self.espacio_acciones = espacio_acciones
        # Para α dinámico
        self.visitas = np.zeros_like(self.qtable, dtype=np.int64)
        self.alpha_offset = alpha_offset
        self.alpha_min = alpha_min
    
    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy."""
        if random.uniform(0, 1) < self.epsilon:
            # Exploración: elige una acción aleatoria
            return random.randint(0, self.espacio_acciones - 1)
        else:
            # Explotación: elige la acción con el valor Q más alto para el estado actual
            return np.argmax(self.qtable[estado, :])
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando alpha dinámico basado en visitas."""
        # Actualizamos epsilon en cada paso
        self.epsilon = max(self.epsilon_min, self.epsilon - self.epsilon_decay)

        # Incrementar contador de visitas
        self.visitas[estado, accion] += 1
        # Calcular α dinámico: α(s,a) = max(α_min, 1/(offset + visitas))
        alpha_sa = max(self.alpha_min, 1.0 / (self.alpha_offset + self.visitas[estado, accion]))
        # Calcular target de TD
        if terminado:
            td_target = recompensa
        else:
            td_target = recompensa + self.gamma * np.max(self.qtable[siguiente_estado, :])
        # Actualizar Q-table con fórmula optimizada: Q = (1-α)*Q_old + α*target
        q_old = self.qtable[estado, accion]
        self.qtable[estado, accion] = (1.0 - alpha_sa) * q_old + alpha_sa * td_target
    
    def get_alpha_promedio(self):
        """Obtiene el valor promedio actual de alpha para análisis."""
        alphas = []
        for s in range(self.qtable.shape[0]):
            for a in range(self.qtable.shape[1]):
                if self.visitas[s, a] > 0:
                    alpha_sa = max(self.alpha_min, 1.0 / (self.alpha_offset + self.visitas[s, a]))
                    alphas.append(alpha_sa)
        return np.mean(alphas) if alphas else self.alpha_min
    
    def mostrar_hiperparametros(self):
        """Muestra los valores actuales de los hiperparámetros del agente."""
        print("Hiperparámetros del Agente Q-Learning")
        print(f"epsilon_decay   : {self.epsilon_decay}")
        print(f"gamma           : {self.gamma}")
        print(f"alpha offset    : {self.alpha_offset}")
        print(f"alpha min       : {self.alpha_min}")

    def mostrar_política(self):
        """Muestra la política aprendida por el agente DQN."""
        acciones_mapa = {0: '←', 1: '↓', 2: '→', 3: '↑'}
        politica = np.array([acciones_mapa[int(np.argmax(self.qtable[s, :]))] 
                          for s in range(self.qtable.shape[0])]).reshape(self.n, self.n)
        print(politica)
        

## 2. **Agente DQN**
La siguiente celda define el agente DQN a implementar. 

In [44]:
class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida):
        super(DQN, self).__init__()
        # Conexiones capa de entrada a oculta
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto)
        # Conexiones capa oculta a salida
        self.fc2 = nn.Linear(tamano_oculto, tamano_salida)
    
    def forward(self, x):
        # Se aplica ReLU a la combinacion lineal que llega a la capa oculta
        x = F.relu(self.fc1(x))
        # Se retorna la combinacion lineal que llega a la capa de salida
        return self.fc2(x)

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""

    def __init__(self, espacio_observacion, espacio_acciones, tamano_oculto=128,
                 lr=0.00025, gamma=0.99, epsilon=1, epsilon_decay=0.99, epsilon_min=0.01,
                 memoria_max=10000, batch_size=64, contador_tope=1000):
        """Inicializa el agente DQN.
        
        Args:
            espacio_observacion (int): Número de estados posibles.
            espacio_acciones (int): Número de acciones posibles.
            tamano_oculto (int): Número de neuronas en la capa oculta.
            lr (float): Tasa de aprendizaje de la red neuronal.
            gamma (float): Factor de descuento.
            epsilon (float): Probabilidad inicial de exploración (epsilon-greedy).
            epsilon_decay (float): Cantidad fija a disminuir epsilon por episodio.
            epsilon_min (float): Valor mínimo que puede tomar epsilon en el entrenamiento.
            memoria_max (int): Tamaño de la memoria de experiencias.
            batch_size (int): Tamaño del lote que se extrae de la memoria de experiencias para cada aprendizaje. 
            contador_tope (int): Cantidad de pasos en los que se actualiza la red objetivo.
        """
        self.n = int(np.sqrt(espacio_observacion))
        # Parametros de la red
        self.tamano_entrada = espacio_observacion
        self.tamano_salida = espacio_acciones
        self.tamano_oculto = tamano_oculto
        self.memoria_max = memoria_max
        self.batch_size = batch_size
        self.lr = lr
        self.contador_tope = contador_tope
        self.counter = self.contador_tope
        # Parámetros de Q-Learning
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.gamma = gamma
        self.epsilon = epsilon
        # Inicialización de la red
        self.model = DQN(self.tamano_entrada, self.tamano_oculto, self.tamano_salida)
        self.target_model = DQN(self.tamano_entrada, self.tamano_oculto, self.tamano_salida)
        self.target_model.load_state_dict(self.model.state_dict())
        self.target_model.eval()
        self.criterion = nn.SmoothL1Loss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.memoria = deque(maxlen=self.memoria_max)

    def actualizar_red_objetivo(self):
        """Cada <self.contador> pasos actualiza los pesos de la red objetivo con los pesos de la red principal."""
        self.counter -= 1
        if self.counter == 0:
            self.target_model.load_state_dict(self.model.state_dict())
            self.counter = self.contador_tope

    def guardar_experiencia(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Guarda la experiencia (s, a, r, s', t) en la memoria del agente"""
        self.memoria.append((estado, accion, recompensa, siguiente_estado, terminado))
    
    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        # Tomamos una acción aleatoria con probabilidad epsilon (exploración)
        if random.random() < self.epsilon:
            accion = np.random.randint(self.tamano_salida)
        # Tomamos la acción con mayor Q-valor con probabilidad 1 - epsilon (explotación)
        else:
            tensor_estado = torch.eye(self.tamano_entrada)[estado].float().unsqueeze(0)
            with torch.no_grad():
                q_valores = self.model(tensor_estado)
                accion = torch.argmax(q_valores).item()
        return accion
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la red neuronal usando la experiencia (s, a, r, s', t)."""
        # Aseguramos que este el grad_enabled (Por alguna razón falla si se cambia de 4x4 a 8x8 sin esta linea)
        torch.set_grad_enabled(True)

        # Si termino el episodio actualizamos epsilon
        if terminado:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

        # Empezamos guardando la experiencia en la memoria
        self.guardar_experiencia(estado, accion, recompensa, siguiente_estado, terminado)
        self.actualizar_red_objetivo()
        # Hasta no tener un batch completo en memoria no empezamos
        if len(self.memoria) < self.batch_size:
            return
        
        # 0 - Limpiamos los gradientes acumulados antes de la actualización
        self.optimizer.zero_grad()
        # 1 - Tomamos un batch aleatorio de la memoria y generamos tensores
        batch = random.sample(self.memoria, self.batch_size)
        estados, acciones, recompensas, siguientes_estados, terminados = zip(*batch)
        estados = torch.FloatTensor(np.identity(self.tamano_entrada)[list(estados)])
        acciones = torch.LongTensor(acciones).unsqueeze(1)
        recompensas = torch.FloatTensor(recompensas).unsqueeze(1)
        siguientes_estados = torch.FloatTensor(np.identity(self.tamano_entrada)[list(siguientes_estados)])
        terminados = torch.FloatTensor(terminados).unsqueeze(1)
        # 2 - Calculamos los Q-valores para estados actuales
        q_valores = self.model(estados)
        q_valor_accion = q_valores.gather(1, acciones)
        # 3 - Calculamos los objetivos (target) usando una red objetivo
        with torch.no_grad():
            q_valores_siguientes = self.target_model(siguientes_estados)
            max_q_valor_siguientes = q_valores_siguientes.max(1, keepdim=True)[0]
            objetivos = recompensas + (1 - terminados) * self.gamma * max_q_valor_siguientes
        # 4 - Calculamos la pérdida entre el Q-valores actuales y los objetivos
        perdida = self.criterion(q_valor_accion, objetivos)
        # 5 - Calculamos los gradientes mediante backpropagation
        perdida.backward()
        # 6 - Actualizamos los parámetros de la red usando el optimizador
        self.optimizer.step()

    def mostrar_hiperparametros(self):
        """Muestra los valores actuales de los hiperparámetros del agente."""
        print("Hiperparámetros del Agente DQN")
        print(f"tamaño_oculto   : {self.tamano_oculto}")
        print(f"memoria_max     : {self.memoria_max}")
        print(f"batch_size      : {self.batch_size}")
        print(f"contador_tope   : {self.contador_tope}")
        print(f"learning_rate   : {self.lr}")
        print(f"epsilon_min     : {self.epsilon_min}")
        print(f"epsilon_decay   : {self.epsilon_decay}")
        print(f"gamma           : {self.gamma}")
    
    def mostrar_política(self):
        """Muestra la política aprendida por el agente DQN."""
        acciones_mapa = {0: '←', 1: '↓', 2: '→', 3: '↑'}
        politica = []
        # Iteramos por todos los estados posibles (se asume entorno discreto)
        for s in range(self.tamano_entrada):
            # Representamos el estado como one-hot vector
            estado_tensor = torch.eye(self.tamano_entrada)[s].float().unsqueeze(0)
            with torch.no_grad():
                q_valores = self.model(estado_tensor)
                accion_optima = torch.argmax(q_valores).item()
            politica.append(acciones_mapa.get(accion_optima, str(accion_optima)))

        politica = np.array(politica).reshape(self.n, self.n)
        print(politica)


## 3. **Entrenamiento y Evaluación**
Las siguientes celdas definen funciones auxiliares para entrenar y evaluar los agentes

In [45]:
def evaluar_agente_custom(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    Asegura que únicamente se hace explotación (seteando epsilon en 0).
    Al finalizar restaura epsilon.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    eps_backup = agente.epsilon
    # No nos interesa explorar en esta etapa
    agente.epsilon = 0.0
    resultados = evaluar_agente(agente, env, num_episodios=num_episodios)
    agente.epsilon = eps_backup
    return resultados

In [46]:
import copy

def entrenar_agente_bloques(agente: AgenteQLearning, env, num_episodios=150000, eval_intermedia=1000,
                            max_pasos=100, tam_bloque=15000, verbose=True):
    """
    Entrena un agente en el entorno utilizando bloques de episodios.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        eval_intermedia: Número de episodios con los que se hace evaluación intermedia
        max_pasos: Máximo de pasos por episodio
        tam_bloque: Tamaño en episodios de cada bloque de entrenamiento
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []

    mejor_tasa = -1.0
    mejor_q = None
    progreso = []
    bloques = max(1, num_episodios // tam_bloque)

    if verbose:
        print(f"Entrenando {num_episodios} episodios en {bloques} bloques de {tam_bloque}")
        print(f"α dinámico: α(s,a) = max({agente.alpha_min}, 1/({agente.alpha_offset} + visitas))")
    
    for b in range(bloques):
        for ep in range(tam_bloque):
            estado, _ = env.reset()
            recompensa_total = 0
            pasos = 0

            for paso in range(max_pasos):
                accion = agente.elegir_accion(estado)
                siguiente_estado, recompensa, terminado, truncado, _info = env.step(accion)

                agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)

                estado = siguiente_estado
                recompensa_total += recompensa
                pasos += 1

                if terminado or truncado:
                    break
        
            recompensas_episodios.append(recompensa_total)
            longitudes_episodios.append(pasos)

        # Evaluación y checkpoint
        res_mid = evaluar_agente_custom(agente, env, num_episodios=eval_intermedia)
        tasa_mid = res_mid['tasa_victorias']
        
        # Calcular α promedio actual
        alpha_promedio = agente.get_alpha_promedio()
        
        progreso.append({
            'bloque': b + 1, 'tasa': tasa_mid, 'epsilon': agente.epsilon, 
            'alpha_promedio': alpha_promedio, 'visitas_total': np.sum(agente.visitas)
        })
        if verbose:
            print(f"Bloque {b+1}/{bloques} | tasa_greedy={tasa_mid:.1%} | ε={agente.epsilon:.4f} | α_prom={alpha_promedio:.4f}")

        if tasa_mid > mejor_tasa:
            mejor_tasa = tasa_mid
            mejor_q = copy.deepcopy(agente.qtable)

    if mejor_q is not None:
        agente.qtable = mejor_q

    return recompensas_episodios, longitudes_episodios, progreso


In [50]:
from itertools import product
import copy

def optimizar_hiperparametros(agente: Agente, param_ranges, env, num_episodios=1000,
                              max_pasos=100, eval_episodios=1000, verbose=True):
    """
    Optimiza los hiperparámetros de un agente de RL (DQN o Q-Learning) usando búsqueda exhaustiva.
    Retorna el mejor agente encontrado y sus resultados promedio de evaluación.
    """
    mejores_resultados = None
    mejor_agente = None
    param_grid = []

    # Generar combinaciones de hiperparámetros
    keys = param_ranges.keys()
    values = ( [round(float(v), 5) for v in param_ranges[k]] for k in keys )
    for combination in product(*values):
        params = dict(zip(keys, combination))
        param_grid.append(params)

    for params in param_grid:
        print(f"Probando configuración: {params}", flush=True)
        # Q-Learning
        if isinstance(agente, AgenteQLearning):
            # Método get permite usar valores por defecto si no se especifican en params
            epsilon = params.get("epsilon", agente.epsilon)
            epsilon_min = params.get("epsilon_min", agente.epsilon_min)
            epsilon_decay = params.get("epsilon_decay", agente.epsilon_decay)
            gamma = params.get("gamma", agente.gamma)
            alpha_offset = params.get("alpha_offset", agente.alpha_offset)
            alpha_min = params.get("alpha_min", agente.alpha_min)
            # Creamos un nuevo agente
            agente_aux = AgenteQLearning(env.observation_space.n, env.action_space.n, gamma=gamma,
                                         alpha_offset=alpha_offset, alpha_min=alpha_min, epsilon=epsilon, 
                                         epsilon_decay=epsilon_decay, epsilon_min=epsilon_min)
            # Evaluamos el agente con la configuración actual
            tam_bloque = int(num_episodios / 10)
            recompensas, l, p = entrenar_agente_bloques(agente_aux, env, num_episodios=num_episodios, eval_intermedia=1000,
                                                         max_pasos=max_pasos, tam_bloque=tam_bloque, verbose=verbose)
            resultados = evaluar_agente_custom(agente_aux, env, num_episodios=eval_episodios)

        # DQN
        elif isinstance(agente, AgenteDQN):
            # Método get permite usar valores por defecto si no se especifican en params
            tamano_oculto = params.get("tamano_oculto", agente.tamano_oculto)
            memoria_max = params.get("memoria_max", agente.memoria_max)
            batch_size = params.get("batch_size", agente.batch_size)
            contador_tope = params.get("contador_tope", agente.contador_tope)
            lr = params.get("lr", agente.lr)
            epsilon = params.get("epsilon", agente.epsilon)
            epsilon_min = params.get("epsilon_min", agente.epsilon_min)
            epsilon_decay = params.get("epsilon_decay", agente.epsilon_decay)
            gamma = params.get("gamma", agente.gamma)
            # Creamos un nuevo agente
            agente_aux = AgenteDQN(env.observation_space.n, env.action_space.n, tamano_oculto=tamano_oculto, lr=lr,
                                    gamma=gamma, epsilon=epsilon, epsilon_decay=epsilon_decay, epsilon_min=epsilon_min,
                                    memoria_max=memoria_max, batch_size=batch_size, contador_tope=contador_tope)
            # Evaluamos el agente con la configuración actual
            recompensas, l = entrenar_agente(agente_aux, env, num_episodios=num_episodios, verbose=verbose)
            resultados = evaluar_agente_custom(agente_aux, env, num_episodios=eval_episodios)

        # Guardamos los mejores resultados
        if (mejores_resultados is None) or (resultados['recompensa_promedio'] > mejores_resultados['recompensa_promedio']):
            mejores_resultados = resultados
            mejor_agente = copy.deepcopy(agente_aux)
        print(f"Recompensa promedio: {resultados['recompensa_promedio']}")

    # TODO
    # Si queremos guardar estadisticas de la optimizacion
    # Estadísticas
    # estadisticas = {
    #     "tiempo_entrenamiento": t,
    #     "mejor_resultado": mejores_resultados,
    #     "progreso": p
    # }

    print("\nMejor configuración encontrada:")
    mejor_agente.mostrar_hiperparametros()
    return mejor_agente, mejores_resultados


Las siguientes celdas ejecutan la optimización de los hiperparámetros para cada uno de los modelos.

Entrenan, evalúan y se quedan con el mejor agente.

In [48]:
param_ranges_ql = {
    "gamma": [0.95, 0.99, 0.999],
    'epsilon_decay': [0.00014, 0.00015, 0.00016],
    'alpha_offset': [15, 25], 
    'alpha_min': [0.01, 0.05],
}

param_ranges_dqn = {
    "tamano_oculto": [128, 256],
    "gamma": [0.95, 0.99, 0.999],
    "epsilon_decay" : [0.85, 0.90, 0.95],
    "lr": [0.0001, 0.00025, 0.0005]
}

### **Q-Learning**

**Entrenamiento**

In [ ]:
env = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=True)
agente_ql = AgenteQLearning(env.observation_space.n, env.action_space.n)
agente_ql, _ = optimizar_hiperparametros(agente_ql, param_ranges_ql, env, num_episodios=10000, max_pasos=150, verbose=False)


Probando configuración: {}
Entrenando 10000 episodios en 10 bloques de 1000
α dinámico: α(s,a) = max(0.03, 1/(25 + visitas))
Bloque 1/10 | tasa_greedy=7.0% | ε=0.8777 | α_prom=0.0300
Bloque 2/10 | tasa_greedy=26.0% | ε=0.7482 | α_prom=0.0300
Bloque 3/10 | tasa_greedy=23.8% | ε=0.5990 | α_prom=0.0300
Bloque 4/10 | tasa_greedy=66.9% | ε=0.4161 | α_prom=0.0300
Bloque 5/10 | tasa_greedy=74.9% | ε=0.1629 | α_prom=0.0300
Bloque 6/10 | tasa_greedy=72.7% | ε=0.0100 | α_prom=0.0300
Bloque 7/10 | tasa_greedy=73.7% | ε=0.0100 | α_prom=0.0300
Bloque 8/10 | tasa_greedy=73.6% | ε=0.0100 | α_prom=0.0300
Bloque 9/10 | tasa_greedy=72.1% | ε=0.0100 | α_prom=0.0300
Bloque 10/10 | tasa_greedy=74.7% | ε=0.0100 | α_prom=0.0300
Recompensa promedio: 0.757

Mejor configuración encontrada:
Hiperparámetros del Agente Q-Learning
epsilon_decay   : 1.5e-05
gamma           : 0.99
alpha offset    : 25
alpha min       : 0.03


**Evaluación**

In [27]:
resultados_ql = evaluar_agente_custom(agente_ql, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados_ql, "Agente Q-Learning - mejores hiperparámetros")
agente_ql.mostrar_política()


Agente Q-Learning - mejores hiperparámetros - Resultados de Evaluación:
Tasa de Victorias: 73.6%
Recompensa Promedio: 0.736
Desviación Estándar: 0.441
Total de Victorias: 3678
[['←' '↑' '←' '↑']
 ['←' '←' '←' '←']
 ['↑' '↓' '←' '←']
 ['←' '→' '↓' '←']]


### **DQN**

**Entrenamiento**

In [49]:
env = gym.make('FrozenLake-v1',map_name="4x4", is_slippery=True)
agente_dqn = AgenteDQN(env.observation_space.n, env.action_space.n)
agente_dqn, _ = optimizar_hiperparametros(agente_dqn, param_ranges_dqn, env, num_episodios=10000, verbose=False)


Probando configuración: {'tamano_oculto': 128.0, 'gamma': 0.95, 'epsilon_decay': 0.85, 'lr': 0.0001}


TypeError: AgenteDQN.__init__() got an unexpected keyword argument 'verbose'

**Evaluación**

In [ ]:
resultados_dqn = evaluar_agente(agente_dqn, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados, "Agente DQN - mejores hiperparámetros")
agente_dqn.mostrar_política()


Agente DQN - mejores hiperparámetros - Resultados de Evaluación:
Tasa de Victorias: 68.3%
Recompensa Promedio: 0.683
Desviación Estándar: 0.465
Total de Victorias: 3413
[['←' '↑' '↑' '↑']
 ['←' '→' '→' '→']
 ['↑' '↓' '←' '↑']
 ['→' '→' '↓' '↑']]


## 4. **Comparación entre modelos**
Las siguientes celdas presentan la comparación en el desempeño entre los modelos de Q-Learning y DQN.

In [ ]:
import pandas as pd

resultados_optimizacion = []

# TODO
# Análisis de resultados
print("\n" + "="*60)
print("📊 RESULTADOS ")
print("="*60)

df_opt = pd.DataFrame(resultados_optimizacion)
df_opt_sorted = df_opt.sort_values('tasa_final', ascending=False)

mejor_opt = df_opt_sorted.iloc[0]
print(f"\n🏆 MEJOR RESULTADO:")
print(f"   {mejor_opt['nombre']}: {mejor_opt['tasa_final']:.1%}")
print(f"   γ={mejor_opt['gamma']}, α_offset={mejor_opt['alpha_offset']}, α_min={mejor_opt['alpha_min']}")
print(f"   Recompensa: {mejor_opt['recompensa']:.3f} ± {mejor_opt['desv_std']:.3f}")


# Gráfico de progreso
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
for i, res in enumerate(resultados_optimizacion):
    prog = res['progreso']
    plt.plot(prog['bloque'] * 15000, prog['tasa'], 
             marker='o', label=res['nombre'], alpha=0.8)
plt.xlabel('Episodios')
plt.ylabel('Tasa greedy')
plt.title('Progreso α=1/visitas optimizado')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 2)
plt.bar(range(len(df_opt)), df_opt['tasa_final'], alpha=0.7)
plt.axhline(y=0.5, color='red', linestyle='--', label='Objetivo 50%')
plt.xticks(range(len(df_opt)), [f"Cfg {i+1}" for i in range(len(df_opt))])
plt.ylabel('Tasa final')
plt.title('Comparación final')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 3)
for i, res in enumerate(resultados_optimizacion):
    prog = res['progreso']
    plt.plot(prog['bloque'] * 15000, prog['alpha_promedio'], 
             marker='.', label=res['nombre'], alpha=0.8)
plt.xlabel('Episodios')
plt.ylabel('α promedio')
plt.title('Evolución de α promedio')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(2, 2, 4)
plt.scatter(df_opt['gamma'], df_opt['tasa_final'], s=150, alpha=0.7)
for i, row in df_opt.iterrows():
    plt.annotate(f"Cfg {i+1}", (row['gamma'], row['tasa_final']), 
                xytext=(5, 5), textcoords='offset points')
plt.xlabel('Gamma')
plt.ylabel('Tasa final')
plt.title('Gamma vs Rendimiento')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📋 TABLA DETALLADA:")
tabla_opt = df_opt_sorted[['nombre', 'tasa_final', 'gamma', 'alpha_offset', 'alpha_min', 'tiempo_s']].round(4)

print(tabla_opt.to_string(index=False))

## Extra. **Entorno 8x8**

### **Q-Learning**

In [33]:
env = gym.make('FrozenLake-v1',map_name="8x8", is_slippery=True)
agente_ql_8 = AgenteQLearning(env.observation_space.n, env.action_space.n, epsilon_decay=0.0000001)
r, l, p = entrenar_agente_bloques(agente_ql_8, env, num_episodios=150000, max_pasos=300, tam_bloque=15000)

Entrenando 150000 episodios en 10 bloques de 15000
α dinámico: α(s,a) = max(0.03, 1/(25 + visitas))
Bloque 1/10 | tasa_greedy=16.1% | ε=0.9516 | α_prom=0.0304
Bloque 2/10 | tasa_greedy=28.9% | ε=0.9008 | α_prom=0.0303
Bloque 3/10 | tasa_greedy=64.0% | ε=0.8471 | α_prom=0.0302
Bloque 4/10 | tasa_greedy=53.6% | ε=0.7909 | α_prom=0.0301
Bloque 5/10 | tasa_greedy=59.8% | ε=0.7314 | α_prom=0.0300
Bloque 6/10 | tasa_greedy=56.0% | ε=0.6689 | α_prom=0.0300
Bloque 7/10 | tasa_greedy=37.6% | ε=0.6028 | α_prom=0.0300
Bloque 8/10 | tasa_greedy=55.8% | ε=0.5330 | α_prom=0.0300
Bloque 9/10 | tasa_greedy=55.4% | ε=0.4583 | α_prom=0.0300
Bloque 10/10 | tasa_greedy=57.6% | ε=0.3784 | α_prom=0.0300


In [34]:
resultados_ql_8 = evaluar_agente_custom(agente_ql_8, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados_ql_8, "Agente Q-Learning - mejores hiperparámetros")
agente_ql_8.mostrar_política()


Agente Q-Learning - mejores hiperparámetros - Resultados de Evaluación:
Tasa de Victorias: 62.8%
Recompensa Promedio: 0.628
Desviación Estándar: 0.483
Total de Victorias: 3138
[['↑' '→' '→' '→' '→' '→' '→' '←']
 ['↑' '↑' '↑' '↑' '↑' '→' '→' '↓']
 ['↑' '↑' '←' '←' '→' '↑' '→' '↓']
 ['↑' '↑' '↑' '↓' '←' '←' '→' '→']
 ['↑' '↑' '←' '←' '→' '↓' '↑' '→']
 ['←' '←' '←' '→' '↑' '←' '←' '→']
 ['←' '←' '←' '↑' '←' '→' '←' '→']
 ['←' '↑' '↓' '←' '↓' '↑' '→' '←']]


### **DQN**

In [ ]:
env = gym.make('FrozenLake-v1',map_name="8x8", is_slippery=True)
agente_dqn_8 = AgenteDQN(env.observation_space.n, env.action_space.n, memoria_max=20000, batch_size=128, lr=0.0002)
r, l = entrenar_agente(agente_dqn_8, env, num_episodios=20000, max_pasos=150)

In [ ]:
resultados_dqn_8 = evaluar_agente_custom(agente_dqn_8, env, num_episodios=5000)
imprimir_resultados_evaluacion(resultados_dqn_8, "Agente Q-Learning - mejores hiperparámetros")
agente_dqn_8.mostrar_política()


Agente Q-Learning - mejores hiperparámetros - Resultados de Evaluación:
Tasa de Victorias: 61.8%
Recompensa Promedio: 0.618
Desviación Estándar: 0.486
Total de Victorias: 3089
[['→' '→' '→' '→' '→' '→' '→' '→']
 ['↑' '↑' '↑' '↑' '↑' '→' '→' '↓']
 ['→' '→' '←' '→' '→' '↑' '→' '↓']
 ['↑' '←' '↑' '↑' '←' '→' '→' '→']
 ['←' '↑' '←' '→' '→' '↓' '↑' '→']
 ['→' '→' '→' '↓' '→' '←' '→' '→']
 ['←' '→' '→' '↑' '→' '→' '→' '→']
 ['←' '→' '→' '→' '→' '↓' '↓' '→']]
